# CFPB v05.2：生成對話 × Seed × EDA 語言比較 v01

只讀取既有 Super 20-row run；不生成新對話，不呼叫 LLM/judge，不需要 API key。
輸出為探索性 Excel / HTML / CSV / 中文摘要，不是品質判決或正式分布校準。
全部 20 筆保留；完整 seed、excerpt、grounding、user、assistant 分開分析。

使用 VS Code Colab 插件連接 CPU runtime，依序 Run All。安裝會下載程式庫與
spaCy 英文模型，但不會將資料傳送到模型 API。報告仍含衍生文本，請留在私人 Drive。
已有本地報告時可以直接閱讀，不必重跑。Colab Python 3.11–3.13；GPU 不需要。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            drive.mount("/content/drive", timeout_ms=180000)
        except Exception as exc:
            raise RuntimeError("Drive authentication failed. Reconnect the Colab runtime and authorize the correct Google account, then rerun. No project input was read.") from exc
    PROJECT_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval project first")

ANALYSIS_ID_OVERRIDE = ""  # Set a NEW explicit ID if desired; never overwrite an existing report.
ANALYSIS_ID = ANALYSIS_ID_OVERRIDE or datetime.now(timezone.utc).strftime("analysis_%Y%m%dT%H%M%S%fZ")
if not ANALYSIS_ID or Path(ANALYSIS_ID).name != ANALYSIS_ID or "/" in ANALYSIS_ID or "\\" in ANALYSIS_ID or ANALYSIS_ID in {".", ".."}:
    raise ValueError("ANALYSIS_ID must be a single directory name")
OUT = (PROJECT_ROOT / "outputs/analysis/smoke_only/cfpb_v052_linguistic_comparison"
       / "source_run_20260919T220844338002Z" / ANALYSIS_ID)
print({"project_root": str(PROJECT_ROOT), "new_analysis_output": str(OUT), "api_calls": False})


## 1. 安裝本地文字分析套件

如安裝後提示已載入套件版本衝突，Restart Session 再從首格執行。
解析器不可用時會停止，不會悄悄改用 regex 假裝已完成句法分析。


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "spacy==3.8.7", "pandas>=2.2,<3", "numpy>=1.26,<3", "pyarrow>=17,<24",
    "scikit-learn>=1.5,<2", "openpyxl>=3.1,<4",
    "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"])


## 2. 載入內嵌、SHA-256 驗證的分析程式與歷史 detector contract


In [ ]:
import base64, gzip, hashlib, importlib.util, tempfile
EMBEDDED = {'cfpb_v052_linguistic_comparison_v01.py': {'sha256': '03701539cecc1bfbf9bdb3d9b6ec51c95d670d500ac0d42e534e3c35e6eed81d', 'gzip_base64': 'H4sIAAAAAAAC/91963cbRbbvd/8VfZp1bySQ5dghwDhx5nryGHJPSHITwzlzhVafttSyG0tqHbXkBx6vFR4hCUlIzgzJEGB4DRMCAwTuDJAJAf6XOZFsf+JfuL+9d1V3datlO7nn02XNxK3uql1Vu3bt2q/aZdv2Kc+tjgbN+ooVBt12xRtrd5uW23TrK6Ef7rO85VY9aLudoL1i1f3mXNcPO34lLFjNoINillupeK2O26x41pzb8YojI2HLPbhitdx26IWW51bmrU633bRCD69Qor5SxMdm1Q3HTq5Mt9vBUsEKF+qe26Zmq1bQ8pqtleW6NY9fdW+k487WPbRX95b9ilu3wo7bkT5wcfQvaHfConU8sOa8pocm/KBpBW3rhW51zrNOH/pnyw8tv0HFvGpxxLbtkVo7aFiOU+uiZ57jqK+Ah0Fx/XBkRL9rz/FQ9O95N5yv+7PRz06jrp/lDz4WG17HrbodV395IQya+rkdgeo2/UpQ9bgg96gS1OtehdvXXToYdJsdr12wql7N7dY7Vb/SkcKo5nX8hqdL6t8Fi/59MWh6Uq7ldqjDuthJ/IwG1+w2WiuWG1rNln4lU0PvWlUBEE2IKvEvQXthNggWkl+LYWcF86QLTdf9uWbDa3YK1pGA/kW7GEfziF+vpyp2O349qjfndRygodtoOnWPakhhRSDFmufylHnLnbbLmCp28JzA1nNAYdD2XySszdT8ai1+MTJy6tnjzqnDx6wpyw66nVa3E47FVDMWNoIFz6HFMFaptWad0POqzuLuvRNOy295IH98XPTabb/qjRkLw2l6jaDTDppO2G15bdSYoFXkTOyeeGL3L8Z/MTMxsfupxx/fs+ep3bsn/rc9cvjQdLoTRARRI0bjsiYdEMkY/k99GY9BPzm+Z2b88b2PT+wB0NOHDx/SUAla6HXGKl1acdUxBtUKgnoooKt+iFa9sWh80qeZ6V8dO3waAEp2vNQxpkbDba8UK+GiXbDsVttbdOseFjx62Az9jr/od6KvHUK40wlaDuauEcrrESvxn11re//eBW04zbm2qwvhtZrctjeHltEiLRu7PHJ6ZvrXqlvc4Vq3XqfyGjfLFa/d6tCbuTYIoIqu8w+ZV1Tohl47+cYNwdzAtToAf/LY9MHDT584dujwKWqk7RUrQaPl171c236+lPvl5Cng5uDM4UO/PXn0qHNw+viho4emZw7/lv+ZfubEs8dn8s+X7fzIyRPHjh78DUCs2hG5xBRlT1oz7a7HGPQX3cqKA0rya75XxZcjbj2kT7PA6zywveCg+pwPzhd9NJBoVwk//myXiNYBU/RnhYAHawGrQbvh1kFb9aDjuPV6sGS0aAJ1u52g4dKM/3sXIDsrTtWr+CFxIwMcc1XCaARkbeTgieNHjv6aB643DhocVUUhe3H3OE9XiwbdAMer01sPPQ8w20seSL1BBfiTWXFP8ani7gT5KCCpMk9ybTCYtrPkN6vBEj5M7CVMz7exDoTMnLbbnCPElPYUrL3lBFRVruE3aWl0/GYFXAhvaOAT1HW95wCRHS/shJkIBD/vUJeISEdRZc7bx1tfKJth1dgCqcNNnhj/RZ45qnf8yD8ftB7DBtd2/bl5bD2tAM9Ba97DPtyquxVvPqhXMXRsepV6FzuHDdxPPztz9MRxXh7cGbv/xkfrf/uof+YTi4nv53uXNv/4Xv+LP/XuXemdf61355X+Bx/2717/zzMv9X7/0sZfP9s88/bmS2/0zn/c+/Lt/vnrvSu312/cQzErIsb/PPOymgV7Yre1/sVr6y/fxWaFWcAYR8FasZYxvP7rP/Rv/enne+/cv3P5/vd/2vjwVv/25c2/vInq/dtX+h+ev//jxf7nr2+eu7z+zp3e1Uso2btys3/pAtYc2DsmIGyB7iz0cv2bV41GacFbhNV/vPY7tdrxFC12qrB5/SLA0UK3Ns6fs6L1HX9Dpza+uyWyyMYnf+q/d5UG+/3d/icX1y+c2/jsPaPBujcHMrM2XrnUv3H7/g8/rb95q//uZ/03/967eG397VctsMziuNV76Zv+nc97N24Bup5MQsOlc/1bN6TW+jdf379zpn/9q96Fyxs3rq1ffQ3T0Tv76caZs+gPUN3/7rv1W18DGVTs2p3emXtGN8B3LGqu7S5htrxl7HijmPUusFQhNoaXNbfh11dGlzyiGDQuMyyDvv/TH3tfvIXOec22j5nELFloFU32zv7ZagWtbp1pz2jx/o+v9e9etWaOjB49dIQw2X/38/6Fv+sX6AwQ1b9+o3fldYte0CBoIi8Rdl/5Yf2dv/Xf+HP/9psb96gPNKCPP+l9dcVoQaRDmiOweW/Z2nzlh/5bt3tnbmy+dAfY3Dx3pffdn3tnvyPiuHoTJLnx6XubH3yPoeKh98MNkM79e/d6d29Si19c6L/1Rv+N86Cq9Ytn+mcvYkrv373Yu42vZzbO/c1ot42157fHaKOmnR7Mbp5kApEW1wHo81fv/3A56oo01vuPH+7f+UKGh0/r3/5+89M/oFXBA1EFSOinVzCtqvlbN9a//thotXftSu/KnzHa+3cuUpfufdZ7573+uQu9K5/1r1y9/yOo8mb/jd9hYaCYItJzn4HKsBz717/r//hq/85Z6tTVm/13L/SvnadV+85761+8KS97Z88Cz0aDjSAgsti49Xr/pdub174DotB679JPvauXe5eur7/5tdWCQNXqWP03Pumd/xZtAqObf3i3992X69e+knGiC+uvfiuLV96bJPLTH9ev3eidvbX5yi1L2AHR9KWz/Yuf9z+4By6D8r07d4hwf2H1LpzrfXV549OvBYkYN9HFe1eF6WzcvIIChM2/ngNVma2gLBcEnjff/+vY5vvf9368aGJo49NrxLD+eg3vMeLNt79e/+QlGjoI9NUr4HfrH90FE1RVbp1f//IPVsvCAsOcSeub18/0P/teGi2PjIxAxLbCeTdHEnN+kvvR9phfKKG/iK8Te5/IkQwtpYptaE/O7Ao2hVw+X5z3lqv+HDaIXF7B4+8kx2RApdfFeuBWwwGIJNTmwH4D4m9TdrdTG33Kzg8ArWdALRlgSQTJW5AASHWDptC0dtZQkRkx1cGwLL/G1Yskc7Ry+QhV7qIXD61gQSrseqovRjNLbb/jSTvcsyp0jjDHhQsWBEiS+Nyw4vtTSsTADg5ONYV9l6UVp+k25VMem6P9fNOmaqkOR4gJgzp65UMurUJ+CwJoHthcsMoXddeoV9gsBeVUIm+NxWWKCgQmkApj6HG5+Bsrv8AmwSpiT0d/w8lIFMD2HXrWczTCw9Bv27maTUAsL6y4LahIDbfp10AkFkGdtFZ142t23pzIFmtqPC6WFFcwLNIV9LCgBaDmlNJoNBHgG0aX1Wn53m0CfVW3wPspibtUYUxDK+jfSkWJfmvtIkJhSAIHmQrGIMotAtukEOiRieTODdH39GtuekwEJVGvkiXK3IqGyjK5uYrC0u6yjIYoW2OvQKYAaHoAjZnRlUt2DZJ8aJeLoMIGqDmeJkUIaZohBEXkEBX2DdZg/dNU1NZkQrfJmPlT3bgzzEeshh9CVq3MZ8x8hNyi24J6XJX2BBvQ/iLCMTGSieMYPU23wQs/1u8GMCAQqODw4Rqtl7TSapdLVKkMtYz5ol3eFhsk0aSRQDB2gAAmlkwMbEdOMTIWvBXChdIjsXChhnYMzZBUKOar5lCIcNBeon0TBwBa3oaoqG6BAZVsKmGXh1OWlNoxRk/T4NMoRZd2gNEWyUTtYVhVazsqJMxnLKXZ8ttMfGNcAw2UtM6eQKajB0vDJ1xsO6Fb8Fr7pGrU4q5hQDXSmDqB5VpVv4YfJAZzC0rrUP1tBT5Z2rJwAPnbkfl2VClzoPg6hI2owsk510ihOtkoJmtSlulJMD6xuygkOmJQD/rAxBO1GKEUVM/kpdpIlopaHiS3QcSecpfGos5rbM36ooJp4ktuYautomZxsAmpDckNgcXQXwYvlq6JhNKilRmykRYE0BFOn8+vFQjBhQhtakeEro2165CssAy4S2Qgdme9uiINLADZoIKlIkyKOSElv2pLW3hNrVG9skaj21zJ8bYeYgrZoJ1bKJAynicc0pcF4SFUk+DzexjCuLf8gpFLb/jXlgLBMz401ObcWKTMCUkePRRi+fJIUtIAGX5zL0L+ItgF7nokD0Lb9Rqw+ojNQssHJtaGSQcRZyVsJXFqyJgaSCGmVdW5yKKWrt6qinypeCwTaLETODwMKGMw/VRDmyDiS8wxBJ3oDqMS2gWRL6rH7+hX1OqWjOCwFgdYcYZ/A+CYRole2yYxM3S9TZCtYbvBJGUiWsNJc3Ex2lpSps+h/4nNO5wqRZRaUPyPJGfwjmzkUQnVcWUY2Woit+23Y+yDzIwUq6GmtJlVU2UIbwQtsbK5uyrZjlYy4XRQ6GJr/RRjmTfOkjnIqJRqi2QTNSqzcNNtt4Wn6D6Vzd2UlirBg77iwjO05IPYVcH8NtupfURbmsZ0F0AkNX8ZI1p02z4ZlBb9oE60Zye28JSGqGoXWVHxUqqhkLFXLaUs2M4ON31QtvTt9NPTak8LLfH1RBTNgI0uVn23Hsx1SZiOVo/Cqf6UxKLBBnUB4YbJzhlwDb1Tv43bb3hhCGMhUWcdlrmohHDnStAkk65YQgugqXw+PaMRABBaBqtuFJhDMk9uCNB2ADt4pKqt2toLENv+17ZbnYrxm+1EHe5AjqCFwTsEkX+DmtHdzG8ruD3bZKsweTejWkp022dV2BVphdBdmsS6ggbsoq5VicUVWYRY5qFwX2zHufTMmmiFO605l5Q8oYP7VDVHXDfquDU1ZQAmF2v8i/D4eMF6omA9lYE7KguVPQfxlXFfJlA5wTuTlPXfrAl6t9vyoM6bMyEY9AuCRA9+SR5I3Ku8OWpiPVqcXY0Y5qQwIDthTMZbWWrJt2XeyoJqtxKX0L/LQ5m2jU2760UV5Fe5IFY3vE7iX1FKN4RbBU5BLoOyoJmh8LVW6ehxA2iEfSLdSgeuGfNrYuq0t6fj8NTiM/8d3p4BSD+SDw/Mk16tbrU8DFccXCpUpTzoksMX9Vjecic0HHeoItg1tqLoa8zwt4OX9P0BaKMULdry4HIl8kyQrVDtjluJKfkhmjKckWtrCaFPSF1JeZGLIUfYThr9jFCCYlyOPUqgOS5fhFupGdLWlQPLKjbcBY/fYAX948wN8j/tIvL8x5m35HktEi+JpzvMrNp6K1f+dvbGyf7PXnr6WQxXGrNB7NE/cWrm6UhV4QKO9t8h8gJboXgPSynX3oAqAkMKBTaoLfAoMWTs1S2/2aT9jr0KNgyEQ6CR6XAfUKhYqlVD5Vm3sqC4abPeIhWfu0fDzSXBiNOyTHtMSAwbYlqTyCOSW1Gfgz5KdtyiMbSkZ7Os9xX4iRmr7Dh35+bkqe41yA2LcAV7rUg8ZpZkXmqBNEGHzCWDykUSO6fAM3xW0xj+mAAfiyGPcY8s1SPS3kgbMSwlhrtRLCYl7QUvUx9LCVe4vIqfxCleNueQut+B+tqk1otuFSonWB2EANZZckZz2P5XiWYmzT6slRPrAtAUbbLXN8eQwbrE+zs1sVeTqWgU8jlv7VcFDOQpeK0irMxmCzVQAZBOs+o2cyWt5gmkkj/pg6AEGGQVCNLyLLsYq5bkbM6ZjY/qMo9Z4/lyvLgkrMnhYlI4ubSJUOQ9iMEhevSIfqI3rS681fTGjGMoEmNmZUdVZRaQly0a0lMFNd06ttmc7LwV6rNRMjLx1xCOQSyjQpE1FIXUhCjSpKgXSD7otzc1vnt3ssNcFCgxylqPSmGaEPM1iwEK+dKiCuxxVCxImGOVtkAzXrCIq9LX2JJPIUUkU7IRcdKM3YiMbVSElq68qNXdORjt8pFtVD7Q8DX0qJZhNRZphxz8bFbgXsWqWpmkQPD2AkkwjD9BKgoxYBoCv+FS9EpCWpKbi5KAJJ7JFIKkMdmVyyWGUc6LsF4NKpFIrTlELhfvAN3SHjXWLsHkAUBMmCXKcEKUmHricSVaza440r0pM9YsR6CjAmiOkL0WsQkCWKBeEHSyTnAL/MbkURp2iT6XJifKZS2+cQV0kqsYQp40psrvKZepV0FFkRnrv6Rq0iYLcUPsP5WuPLFSWlD/Fyz5HCoGBY4VEKP/iSmK+5sxV0l5nskCoKKBDaGIfDlRTXgBdbDDbTgR7gQgvePJ57cDzCEFjdm5QCvys1Ok6J62WtOdaE2zzNEpwurmiCp0HPyZOPXJUydO8sNzh0/9iv5OH/qf8uc5e42ZBSvTzHgQ01VOqX4xVoePn6iT+5mou4QZdIRRTJksOlFIzy8PMRyKsFAhrEjlQ23KS6MuRkmYQiOL1xxIpfqDgLec/Uuby0p4YdRM1KVkT1skwi16qm6nWPVaCtPwb86+QJ9Z8+wu8+MasWv4BPHIc1RsBO3WvOgLzwU+Ke7pGUy210R8ioSJJlokYRKf7K0ry/rhaK1YdRqYPpvnj3Qd4WwDapXUSKtVQ+RlQ9mSerGyFWtV8mVAq5LX/GM4fOY7Sh9i0iBlKKYyfIl/sN1MsQ79VTZ4PblDmyEg2HKhzunChKFolzTbSEHEpr8DqCqEbDhEPbYtoBliUzQ8IhKqnpQRII+Qvk5rR8kHGWtseEOaCh2KEYMkiXYiwiwYnzEwCAmJYRnl4hFuMSRZX7qhLIiqyM7ggaKbna2gpRYVlx9cVjvsPAKDUWKLdpgxUzvgx8cfuhkXnjQI9LPbNcR8KWL4z/6r4vTEjzqD/GgGPcjiRzvrFNQMzYO36BY4jIz/mUMPO/pK3e2GCQLJXqFp/uxWFyscTFwhwZEelvWDKx/A4fCwltGvB1jhyh5Aji/qW8Vrdracnp1t0A+Hqmi/i/qOriQ3wUJGqcx+p6slUbIFXZDaNrHXIeM8gm4V2xNNYDiPb1EXaLeRowFDMUnaDRfOxRwtKTMwe0swQ+zTORuorQorJDFClKQWXJTDR7LT/3S3FLyH7VY+n7S7PmIdClhGA4GRF7FJnkSJOYZBOICZY8FjxcZvdoNuaIUrzY67TOLHM9MzM6eUShoWU4Id2wU6KyR3iYINCZNdKgPy5TBBNXMrScmPcKw02XZdWuTSiwwwalwZ/kEUfgiyyC2m5DaRY0qZtERiZ1KFV63l86KDStOm/jkUdBRU74iQUVYiq4Y4klbilO5a0AqqihRjlXLQI5ZuU2KPHbJlKUBlZd4nsVbBgazrtqHcI4R1Drp0XdCtfxUsJ96+8yODvsZYzqWBsPaoAZNQgIptQyrIFguGDiAOhTYHQQY4vQPQgAZMCwbrypiLKHxFfiaLkGVRa2urD3MIZG1wOpQ2npNh55MauWBgMpMnDKokk0N5Bwt+osBlL69wey0wS2NiPTjLQ7KdmP+AXI7FbvEzA4RPZkzeiZS4Ha7lhwKEts4j5/i02BjTylosHEukSV5Usi2QQJy7GzsKh+pGwybQ6OFD43EAMw+GWNuUXXi8bBcWMoBvsOrTaT+HTP6EayLr4guIyMnRwB90r4oF9XmECiTh1exVIc01a/SAhWcqIi/sFJWm8PzAveB4IjiwO9F5mGigcQAlHaTaB1siDg+OEVunUA4+iFrFuSQ5FGEnXSiI3TgEr8gRnADyFPMgQSllO0qU0hOfLyRsHAVllFKWSjr+suh7SzltqlTkxEzCJztZqRJv6roQqJiOeDrVzgpCgXM4aUQHeaZs1JoFf8rrXa9iRi9Ee4KieolFSwZfyNEZNtVPHzvmHP7Xk8dOnJqeOXHqN+z8UIFVUTcSentRAkYg5sQrQvrJ27OuJB4caoYmOd2KbKa6bCm7JarJD+VBW5sENYmVRdouEitvza7k1Ooq8DBUMPjksH2XIKhJGFzgHHcectzzUtTJcrGKY1ZNNzfIrgTVMQsYcC7T34zlryDjjZYGtlsOtj5wRluyIj9byRiO+U2GsP36smmBiA8Y5UUKIpgND8EXiQ/0O7cDgK2JvXE1HBKEWAlLexHuFnKnP5n58cm9+S0WpKBXuxvohI5BpMnlpya8Mh9g7fJhUMO8U8g0zWQYIQrZOunAEdW00l7IUrAL2YpMIVvfK6sQPGWhToZNaUN0tGxi0o/iwTKJ368uAxxVB2tRO45eL/nEIkPM/ywziNyWx2gxl7ktj9YOI5TclpJXvrzlgpVpzZCKh2zA2aEeQEYR51wrOCCwY9NkfCg5aDh6GbssykQ/Zx9iReMsL8boqV5RpwBUMxwyL9Q6HJmgv84aX7eBDNdtx82uCjdjRnNqBdZozYFaknsisRrxllYjX5KxXTHGmUE6UbAqHXYa0eF8TM62lhiSXgEJCRRPQD7pViltSS5lM14fHZPlsoimUkf6xXPgKJFxqm3nftnNPz/7/FLp+aVdo+VHn58FdJaraY2o5SN9b6/E9IZ13PZpJS1CA+o4HBlBsTQca2EoOxVELjd5A4Ha6IYUibiSk8qlSTbQQrwoFxtwn/mtevRJf5lEICepXe6yH06NIyywjUNU9VwcwYmovjhKjLIw4OWkGQtnezi8t2ItBhV3Fuc32yu2jnBDVAfFHWZFnqWGWTCGcRyJIuAoEyW4DEet7uqIuJgk6UYGw2II2u3nkz9OATWFCGYjBYvP5UrsbgZ9mLylzLLZkGJpjhJPi+rncE4RhxBp1iweMxbDtP2fe7vFPshBQVsAoM8c0gwhlrftQejWf7ek1PBGFO+lujiSRU2JPBxrzBngEgPYwhIJ+C8gZYvbrj4AROu3+t2WgCX3g1CBQ8yg4zy+myTmLh09JjzIty1AxNNEkr6gOKmR6BMC0rFR6di2c7YNNKbPUUWuW0LDWiOWbh9TC4O0gLqL2ITjJ2ZgbeP4PBgKMP91Pm+N5TIPH0u3QhEPFGpTxYxK4pKUosLMOaWJKLKOjze2PKZ/yVcQGqZWWXOPYDlH59D12m2OcpYNxNNQpEhXMp0k0glQdCkHGDY9DMeqtIMQGWAiHWmM+icmQk7v8CIv5lSulZz+NMWSGEWd6XCfKYo3U4Jyklsz89lO7Ewxb/QyTucwlaNkDuCjs12/DouO6oNiqEEFCVDajCCRsCAa8nalsuqQOGzGOuCrca5QzohEKE5GB8gMUIGozWgLRM2SNoSU8yl+TFsaomK5jIgXEAYEWlL7MDrP4aTWY1PWeKJENCIJN0VQlQKrzRL5hMS5OqCkSLN4IQ8FSSySyH4RnWZJTZNtdA/FzM7mFHg1qLI+yOBXM9eenJxJg9frTKWKQTIGUf31EmD7Emn/yO3EBgCdvASLbYWOeuHYVyUd4U0Tl+pdQY4KqRAQxqZpdYqGbx3AcRTZAtVRDclHo/RSjcpBFBaSqDKRUVDDLG8tnBWiwyHq7yDbKBIuHdG9cqXtu6B6i6nByWuPGe5USa0u9Ydy1ZQZB0w/otxTW4oXsblGUjA5pLepSK3ooFFay9ki0obCviU4NivuKYpRLqdWEsXRyrch0bTi1pRAjqhkFJmbZYGHxKaUKbPdEgX7jZclhB1PiCMkuhjoniCJeNoAaNinXFIC+aAgtaLIlgqrGHu8HQxA3rE+NGiQjPDgEJ2QtkB93061iBx6nIQI5rhQHGwqdZEBlE7sTGbgdVs9K6BT/bTF885MnTPa0Yh6UCjSG4XFOPqagoE06uXMwXZHsuARBseBtY0z8pyYcaZPnz6M/x2Klive/y+FJYuwZM173TZnB1JJ8WAU8rzK/ChWhwW/NFCGLE/t4paGycQ6TxwCS81jYdgcDc7OTi2wW05JIRvXBQNRER/TcgoFVosk50j2PnUkMTbspCI56QS6eSwxTkDAn+cxqkBsqvooHlKG6ZPvQ1KV5dNV9WNJP+hsckU635PLCv+EJgdT80ouCQsxSog+b7NUy74sW3+BXYkytMEMVJkn8aTqcJIbWx2zjLKlZQ9ki8xq+QwIxq+SWf4BR2WCsdmyPjCmOFWQCNI6j5VD9u/QweRjntsIFwntIdz//wfrNFvktjJPZ5ikdfaFrGmYHOqY5bNcBDvhDeZcC2w9yPDpJveFB7dSs/dtG86og8l0iEgzISVyh+VALzNJHQZufBvZzgMUrarJbdcUKBGvYMvlFHpseN0R11MkPklWtLCChDZ8kJRIGJuH8Dw+W7qPnIeSLWwelNHyRIRJMXHdyYK5jApZBjYd3k5nYpwllUJTZc4RJqnIZ2kWFKBzbKo1ujQLQm1A18zhiYQunZokdpyK+EixQAxsMM6AlySqV9okQOOsqUenSlAJ8dXjhiklpiOxBLJ8qSXPpEVZqUhShIIGOjAQslcLRuhIW8NHVvRSyyPyxJRYCCIXb5UjPRDoIat1kQchgrhxBHOxQLY3Ngb5FV00DiBBp8ppOZH8wEkIfGqTBC/2qEC8t/ZMPPnEkzEY5WXJkBIzj+PWJW5/KejWq0icB+NilwNyqvsAjI66IDnp6efGnp555hi96ID128lVHONdNW3OCNQg70WEYLlitLN/NWEnJgzaj4NEN5SZAueVaaLDYhXngZqcvjEtbctsJoXtxDyToWcy3TuVljUGWxpI1przsbcs+VXOgfLEXo36ZWGB3Ch1YVl2Ai1IJPw1JEiQNwfZXskgqKXqiScHuuMhdHUcRy7yxVrAbJMSzeZw9Kw6JdIqWgxgjjjC/w1i2wTg86F/I0ctuUDqLIDV5g4KnPEjjx9+8ik7cwkshYO8n+ArghykoiRJU9GiSok1eMY62h6oFCeKJdsgkQHix8nmw1Ybb9mrdCmPhGRxjjMzMGFSSl/qFQzTbnEkE7KrU/YCcpS+N7fUdlvMWBVS0RRr2VOwcLTsiD1RMjCVWkbbqpoUgEz5kfUZHpHOfCOZs1bDw+hET1u5E/b/E+yGPEyCcGA//YsMH9BTd704P/o0ZNxdB/bTiTt4juiMW2dqF6cCw9sOMpd5B46hbJdUSgrnRKs4nL5/TD6ltwh7P+cuPjAbVFdWiZYmx59oLSNWDsu0Mdr19zXc5VGm6snxvXt3t5bxoo1Aq0ladvtasLuAUicnvMYaNpDVJWwX3iif05rE71FC4D4yEyIebYl/TWJNLM0jEc0+bsxv4tlHQEWqV8zIV2dh5/Dao5Qa2m2F3qR+2Ec5fOvuyuQsfD0LcQPL3Ku1TrXQmVeVJ8dpNETO1iPVajXqcXEvupxulZZ5uKqBzyHV8T76ZxSoaNHh0VHFIibFEgP723itnd83h2GNA1woxs3VWAWYfKS2t/ZE7amo2fGMVv8HO6FzMaJ/sRt4zq8me8NDXVvbPybztX9+/ACy/W3cOrN+4e+9H7+RvJPWP868afVeOStpUPePodDghHfrB0ikspUpyt5f9/kFkVlR8rTlKpxvbv8Yf4oCOXS+VfWRAA1An584sPHJ7zYu/B/Jw4n8hv3/eGvj5ks/3zsfJULllJOv/3zvAno4MQAjWiqlXPQ4KKsOSLl5WO/j8glFoDSSfe7gv8qBnu0KJ10DWxNzgYRUIDhmnpLPwt90lMAV6WI3Pj4nKVqBQCRSRSIXyepIKTnv3d288Z0krx2GTMVjSupvUR1zmxqSEJpjnHITu/Pb97ysnWRIAelwLgXtiDJ98Tr1UJzCReLC8tsZx5gdaqmgxnhZTVAoGX12Kfi7yvk1Hv7+VlYpdeCGSkEozyjA524ESOtAInlYshOtA6Dk/tcvS4pRpHi1VgXAQPKCXeU1yuJ6++bm2+/oQqkMBihC2YElbejnN3tXz/98721V1ExlIOUkKSsIAuVk+iXZqyT7XH/jHMps1Xt7P6KLxSQztYs4y64Dqe18h97pYbnEJ1OnwpKYU7wRfGvPgVVuiSZsDyas7Q1wHyS6fL45OjoKnwy2b1Y28Ite6lC/rFOhiisRQHBJ1d5wdIwBH0Qvz2JQcgeEMp2pM3tYb4LnlBMrWm+TVrrf0WIopUySiMgYcMWVdX8PJJNd2dEoucOqFEsCtpYvkLBSJxtHhqrgBYqqEzMTgoqwztqxlckRNWt4xkyuoy1QJjAjfWZBvirg6byaKrF6nFAzyuyvOzlwuQAnXDKMV7GgEqV201CVhx8gd5Bt1J5WLeo4wQZSkJBO4oKWliAvwVNmBTXr/6WDUeID6hJYHlKi5+LT5fSS1EHgKWG1ka5CvPYOc5U4Yx8IA8cnO/MI9Vxm58WccQUJYoMQHKS6P318+thvTh897Rw9lFKkkKIJmMF1DKncZxTUMTx/al75dIRSEtn/EuSjyvFR7qmdZFwz8kokEmiIl1ayX6t7V5DhAZIpCe8URwskSFlyEnSb7iJ8ySQKitge21AjzyQffWahmhYVBf9P7fAc/UhC1kC1waBWdYBaEidWzItYpgbj9aQHKs+h7Lbi8Rjmtk7YVVNGFPFY0ZQ8gD05miVMdi1Iu9NKqwuTylaA3H6s51dMd99CnC1KGGvByJGzthZLgYxNZTSRLvF52l93EadM/lSz0dVImZ2MpMc1MpQdlG4qqlVOHO73QDjiacAlthpmwC8l/EBJJ5Bx7HbImdv4wG2Gzvnoo6uIG9h+KzL2IR3m9fCb6FpWWq405gcwdAqeM23/lwOvWtcUPGtaieP36NNJpmDHoG98N6k93YyK+3BU3AfbPpUqi6wrSTqPHOuDYA4ixFwHtdMs8PqN3EoO+7HErUqOuyGu1gGo5DNRBmPCKtWN1xZ/NSz5bLvnXFbGmrNPu1DzvDQUtRQzG5w5gosAJCiKfXCmYyOZTJp2xaz7YbIHInfD7Bhw1p0yeRHlKUPJmoqPmZZoDasGNIKDiDEUW8vxw/+it0vsWhzVIiEHFsdpSjLZMcWXWSRlLBZ1PH6xsUC7ndqdla2E9zInWBBlQZhtE8r7PGfhJJFjLAr30h/sRDEFVurmospjIos4ZPPD/VX5IhlZdap1yUafKpDIUz8ATbsZNBtV6W0HASa3xAyoD2CQFosjdCKaUEEG26OjU1CPkTWC6cNKamDJ7O+joT8X5cBMmtcVijNlGEqku1wPl+3I/i5o2a4OSaF2Iq39A5q6stLXS4pEuTSMdxIhTHDKToU4krrbC7m9lnL6ei9cnFXJQ7kPRFHi2H5FTJBtyBuucrqPZB1Hrvm8H3GCKtooYB5in7dcYkQ7BGd2pryWnjAhLV0Ngosu/RGHis5jKel9IxosDJBZsmCStjKagUltQWWrW21OZly3VlT5rHJNld9HJftuKS+13G3GTysu3TzH1ueKv+B3Rvl+MQ6VUfeSYSPKwpzh8VJcWLnV+MiQ+hJJT+Qmr0uOq9DRYaE6nGIkwxEfq7yRc428gZFkUDSLKGdgBiBJOEZ4IWRB6NEJyvhQ/0I+zmSM85pYczRwnZ6M5smVHTuVRG1Nc9HnWJ4WpgjD8pxLcXDMJ+OL2cZkMe6LTqUSPaIw4v3b+srBopl8mfNYC18fSLMvMXesxZlZ/iOi1Nwlv7VepHKCK0eiVUXcRkLTgKGRQmOjN1bqUJri4filMq8oI92D2OjUMRCPk+ORqfsRdYuKmDLFiAkrSP/KlY2fvsINJuvfXO2/9250CY1c2kT3w3x9l/NJpqe/ZouFEXmkIyIlQ4x5cUvvjfd7n1yUcikDjLW6LcGtjZmg+QKVjH7YjzyC+4G+J2sRX2w0aP3E3Tp05861O7AkMIg4YMA4GvJfetDHMNFEDkiezxL/q22mNHPR4bDBczUlfYiqbGSSowmNTCujVurss1hKkIMwPlkZrk2tSicgNk8WJ2rqTGUYZ53K501qeYzIhUYGzPbfPYNLdjZuknXZuO4qtpNeuU0XbV1/PbqcqffVuf7nHwLbdPvW6x+sX/tWjKeZ+Fd+rYcyne7Nm85gc0kmcFQDkv5tlTK0C/S1fyPbH/1Oxi+ujWF99M7E1GoPRQru9dr48szmB6/17l4RmrTQyOYHV/vvvr/x5eu4AAlN9M6eX3/3IqegHnKfGFAqVyf9jCvT9NVJchGSWoZf3Om980H/+r3+rQ97712M7y0yyX/U6v14tv/ZRxs3r0dtU8NRO2iTA8+pubOXcU0SmsA6odui1H1Ql+QaqPW/XOx98De58Wnjo7/QbW241unl27DBMdh3cM0YbjjCXV0bd/+iJpstdMM6hm73//hT3KWYfnBFmlqhipAu9V67jCKDF1mhWTHN9758q3/97+tffrj50u9xn9rGF1+B9jb4Wi991xSNkCcBN1iBlw3r1vqrH+I+J9xR1f/iY21iVDc8YSbkni88wOh7//tv1999n0y/Zz6JLn/CVVmbNz6mG7LY5Nz78qP+u5fv3/0dXRbFN1oBa7j8Se61UhfUUfWXDdo3iUrWsFLeeVXoZ8MPVE5IjYaoqHRR58X5YqOaFBhj86Y0t9XFRqZkaNw6QkJi8nIFfNrR3QqkrLTn6gHyijxq59XZeshuJKXpRIfx1U5qXJGEl77KR3ctYb6Nrr/FRbM13LNLEoYjR8MdQIBzyLEjh68cDdLX7Ban23McAnRSzGVRkE7QnHLoLAkESaMmZxJ1VRVw3VFlvB0lQYIke3iQp06K9VflRDWUsSEwMOZRaF0PW12E3FEtzm4LBnVDNqYxNP5D8MJcKtN+wuJNBYpJsze/imzf/Cslb+sLSGiDN+7iIvlQWiktlE3JMAofNwTcQqaUGmsQkJqjG7zy+ZH/CzY3ciphegAA'}, 'cfpb_v051_legacy_linguistic_contract_v01.json': {'sha256': 'b243de8843cc3bdc504e5a6339547d3eef19d4631657ef3f1acd798bd0a4f210', 'gzip_base64': 'H4sIAAAAAAAC/5VUTW/cOAy951cYuWxTBKknieejPRQ7+wHktofdU10MaJu2hdiSIUrjOlH++z7ZaaenYnKwSFkkxUc+6vkiSS6PbEUZffkRapqtDl6XLemGq0PH31RJ3aEkXamKHMvhmK4ur6OXGG9Ljk5iyw+1goUM3jEfqftQ1kNxEEaIxezAFcE1uxmmn70P0tJtto5B1uvsfrvZrrYZc0m7Ilul2d2mSovNJluXu/W22Nzd7uoNF+nufsPbdbFb3WYp7zarXbpLyyWs0o7tYNmRe0X0n0Y+c+5V8gonOcH5lFj2giOjuykhSShplThjZ7ueSbzlnrVLCquqhm+WawZyuEcLLnjGHn+6bwfNzXzrqVo/jk8uMaU8L959/qhN0MYFzSg/VuXaKI0NI1TjXUCYaADxG1bjuyoqiDwLw7LIuI40b0a2HKWad7RsxsXiu39LR15kNLpCLjOkOce6oyZiuruf/7xc/8DWctUo3ZwJ7SFxrdKP4SEpuFMACK3xLBJ6mgoOvWpaF/UwGBFVdFCsKWhW2LY0SKxNEosfwKJePgcaBiYLJXbXTcsZGlaTjX17SB61Gc+FQx4lturpLf169fkiT1+x48BVmN2vYsq9kjhCaJMWsCVYLk2j1ROwUsWxHJLAC3NlGw4DuI/6c3CWtFA5hzk3dxZQ8y2J//H3P3sk1Hh4gV44NlbzFDoaJ1AOQrxyoQN9u2RJJojnsOcYKNl7URq9g2KZfNjv96DrKJiOShEsAf+oxNizAfTmDdnX1gvK5PJ8fB8wg6r0nfES8EDZKfhB2AUYIMHZAo/QaeM1lSUPDsTiUHuromOFB0rca0DU0sZG1pZUdS4Ax/1g7M/P4i8RTCz4UYHtzsS1w/WRDCPzY+iNdm2YQO2g41Rche9cOR1fBWpMKLg2kfI1goWxZTRJ6ZIB0qkuEg68w/wYneT5lz9///evPP96LiLLAyl79nTjXdRhEXhmSuepw81oUnzCSmORzMwiba7zXN4vljGXgA+DjfXmOb1epy/QCu9+meVF/F4u/gebfO1BqgYAAA=='}}

RUNTIME_SOURCE = Path(tempfile.mkdtemp(prefix="fde_linguistic_v01_"))
for name, item in EMBEDDED.items():
    data = gzip.decompress(base64.b64decode(item["gzip_base64"]))
    assert hashlib.sha256(data).hexdigest() == item["sha256"], name
    (RUNTIME_SOURCE / name).write_bytes(data)
spec = importlib.util.spec_from_file_location("fde_linguistic_v01", RUNTIME_SOURCE / "cfpb_v052_linguistic_comparison_v01.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
CONTRACT = RUNTIME_SOURCE / "cfpb_v051_legacy_linguistic_contract_v01.json"
print({"embedded_hashes_verified": True, "analysis_version": analysis.CONFIG["analysis_version"]})


## 3. 只讀 preflight：來源 hash 與 20 筆配對

需要既有 Super run（包含 evidence manifest 所列全部檔案）、Seed v05.2 parquet / generation JSONL / manifest，
以及 EDA v05.1 manifest 和五份小型背景統計表。缺檔先按操作說明同步，不必重跑 EDA 或 generation。


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, FileLink
inventory, raw, prepared = analysis.verify_inputs(PROJECT_ROOT)
cases = analysis.assemble_cases(PROJECT_ROOT, raw, prepared)
print({"verified_input_files": len(inventory), "cases": len(cases),
       "format_valid_cases": sum(c["format_valid"] for c in cases), "all_cases_retained": True})
display(pd.DataFrame([{k: c[k] for k in ["seed_id", "release_split", "expected_messages", "actual_messages", "format_valid"]} for c in cases]))


## 4. 完整離線分析與匯出

將建立新 analysis 目錄；若重跑此格遇到既有結果，請直接看下一格，或回首格建立新 ANALYSIS_ID。
不刪除／覆寫任何舊輸出。通常數分鐘內完成，CPU 即可。


In [ ]:
report = analysis.run_analysis(PROJECT_ROOT, OUT, CONTRACT)
print({k: report[k] for k in ["cases", "stage_rows", "format_valid_cases", "policy"]})


## 5. 閱讀結果：先 HTML，再 Excel 的逐筆配對，最後才對照 EDA 背景


In [ ]:
report = analysis.read_json(OUT / "analysis_manifest.json")
for name, expected in report["outputs"].items():
    assert analysis.sha(OUT / name) == expected, name
display(Markdown((OUT / "comparison_summary_zh.md").read_text(encoding="utf-8")))
for name in ["linguistic_comparison_20.html", "linguistic_comparison_20.xlsx", "analysis_manifest.json"]:
    print(str(OUT / name))
    display(FileLink(str(OUT / name)))
# If links do not open in VS Code, download the files from this Drive output directory.


## 解讀邊界

- 來源截斷／脫敏與模型改寫分開；不要把 user+assistant 合起來跟投訴敘事比較。
- MATTR 固定 25 tokens，短 turn 留空；否定、情態、候選情緒詞都不是語意真值。
- 舊 EDA 的人口、enrichment、去重／family 權重视角分開；本批 20 筆不是比例樣本。
- 詞彙 overlap / TF-IDF 不是 hallucination 或 correctness；候選修復詞不是已驗證的 conversational repair。
- privacy_verified、benchmark_eligible、formal_pilot_allowed 仍為 False。
